A notebook to compute average "partisan bias" scores by state & chamber

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from knobs_functions import *
import warnings

warnings.filterwarnings('ignore')

Calculate the average value of "partisan bias" metrics by state & chamber
Note: Switch the list of ensembles for just the A0 table

In [2]:
from typing import List, Dict, Tuple, Any
from fetch import _score_mapping

metrics: List[str] = ["disproportionality", "efficiency_gap", "geometric_seats_bias", "seats_bias", "votes_bias", "mean_median_average_district", "lopsided_outcomes", "declination"]
additions: Dict[str, str] = dict(zip(metrics, metrics))
_score_mapping.update(additions)
# ensembles = ["base0", "pop_minus", "pop_plus", "distpair", "ust", "distpair_ust", "reversible", "county25", "county50", "county75", "county100"]
ensembles = ["base0"]


bias_table: Dict[Tuple[str, str], Any] = dict()

for state, chamber in state_chamber_list:
    bias_table[(state, chamber)] = dict()
    for m in metrics:
        all_values: List[float] = []
        for e in ensembles:
            arr = fetch_score_array(state, chamber, e, m)
            all_values.extend(arr)
        # Guard for undefined declinations
        all_values = [x for x in all_values if not np.isnan(x)]
        mean_value = np.mean(all_values)
        bias_table[(state, chamber)][m] = mean_value

bias_table


{('FL', 'congress'): {'disproportionality': 0.051028159999999996,
  'efficiency_gap': 0.03467835,
  'geometric_seats_bias': 0.018314505,
  'seats_bias': 0.02229945,
  'votes_bias': 0.00701748,
  'mean_median_average_district': 0.0223224,
  'lopsided_outcomes': 0.005327065000000001,
  'declination': 6.014968755},
 ('FL', 'upper'): {'disproportionality': 0.035662990000000006,
  'efficiency_gap': 0.01931336,
  'geometric_seats_bias': 0.0027414849999999997,
  'seats_bias': 0.005403224999999999,
  'votes_bias': 0.001652205,
  'mean_median_average_district': 0.012550255000000001,
  'lopsided_outcomes': 0.00183253,
  'declination': 4.25327322},
 ('FL', 'lower'): {'disproportionality': 0.03896679,
  'efficiency_gap': 0.02261685,
  'geometric_seats_bias': 0.01420281,
  'seats_bias': 0.016913074999999996,
  'votes_bias': 0.006509409999999999,
  'mean_median_average_district': 0.020347115000000002,
  'lopsided_outcomes': 0.01151052,
  'declination': 6.922669695},
 ('IL', 'congress'): {'disproport

Convert the dict to a pandas DataFrame and LaTex

TODO's
* Need to generate the LaTex columns as right-justified
* Except the header's which should be centered

I tweaked both by hand

In [3]:
def make_partisan_bias_table(*, latex_filename = None, rounding: int = 2):
    """ This is modeled after mean_diff_table() """

    index_list = [f'{a[0]} {a[1]}' for a in state_chamber_list]
    df = pd.DataFrame(columns = metrics, index = index_list)

    for state, chamber in state_chamber_list:
        for m in metrics:
            multiplier = 1 if m == "declination" else 100
            df.loc[f'{state} {chamber}', m] = bias_table[(state, chamber)][m] * multiplier
    df = df.applymap(pd.to_numeric)
    df = df.round(rounding)
    df_latex = df.copy()
    df_latex = df_latex.applymap(lambda x: f"{x:.2f}") # round values

    # combine the values and markings into dataframes to return and for Latex
    state_chamber_size_dict = {f'{state} {chamber}': f'{state} {num_seats_dict[(state, chamber)]}' 
                            for state, chamber in state_chamber_list}
    for state, chamber in state_chamber_list:
        for m in metrics:
            val = df.loc[f'{state} {chamber}', m]
            df_latex.loc[f'{state} {chamber}', m] = f'\\textcolor{{black}}{{ {val:.2f} }}' # TODO

    greek = {
        'alpha': 'α',
        'beta': 'β', 
        'delta': 'δ'
    }

    metrics_name_dict: Dict[str, str] = {
        "disproportionality": "PR", 
        "efficiency_gap": "EG",
        "geometric_seats_bias": greek['beta'],
        "seats_bias": greek['alpha'] + "_s",
        "votes_bias": greek['alpha'] + "_v",
        "mean_median_average_district": "mM", 
        "lopsided_outcomes": "LO", 
        "declination": greek['delta']
        }

    if latex_filename is not None:
        df_latex.rename(columns=metrics_name_dict, index=state_chamber_size_dict, inplace=True)
        df_latex.to_latex(latex_filename, escape=False)

    return df

Note: Switch the output location for the LaTeX for the A0 table

In [ ]:
# df = make_partisan_bias_table(latex_filename='latex tables/partisan_bias_table.tex')
df = make_partisan_bias_table(latex_filename='latex tables/partisan_bias_table_A0.tex')
df

,disproportionality,efficiency_gap,geometric_seats_bias,seats_bias,votes_bias,mean_median_average_district,lopsided_outcomes,declination
FL congress,5.10,3.47,1.83,2.23,0.70,2.23,0.53,6.01
FL upper,3.57,1.93,0.27,0.54,0.17,1.26,0.18,4.25
FL lower,3.90,2.26,1.42,1.69,0.65,2.03,1.15,6.92
IL congress,-7.13,1.04,3.64,5.63,1.99,2.91,10.67,6.88
IL upper,-7.97,0.20,3.04,5.55,1.94,3.17,9.97,3.02
IL lower,-7.84,0.33,2.73,4.21,1.54,3.03,9.83,1.70
MI congress,3.63,5.52,8.02,8.34,2.27,3.64,5.12,11.84
MI upper,4.36,6.24,7.10,7.15,2.71,3.95,6.03,13.68
MI lower,4.36,6.24,6.71,6.73,2.82,4.27,6.79,14.67
NC congress,4.70,4.13,3.15,3.19,0.78,0.98,0.42,4.32
